In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import kagglehub

In [ ]:
# 2. DATASET DOWNLOAD (FER-2013)
# Downloads directly to the Colab environment
path = kagglehub.dataset_download("msambare/fer2013")
train_dir = os.path.join(path, 'train')
test_dir = os.path.join(path, 'test')

print("Dataset Path:", path)

Using Colab cache for faster access to the 'fer2013' dataset.
Dataset Path: /kaggle/input/fer2013


In [ ]:
# 3. HYPERPARAMETERS
img_size = 224      # Required size for MobileNetV2 Transfer Learning
batch_size = 32     # Optimized for T4 GPU memory
num_classes = 7     # Angry, Disgust, Fear, Happy, Neutral, Sad, Surprise
epochs = 40
learning_rate = 0.001

In [ ]:
# 4. DATA PREPROCESSING & AUGMENTATION
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False  # Important for Confusion Matrix accuracy
)

Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


In [ ]:
# 5. MODEL BUILDING (TRANSFER LEARNING)
# Concept: We use the pre-trained 'imagenet' weights
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(img_size, img_size, 3))

# Adding custom classification layers (the 'head')
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Compile for GPU execution
model.compile(
    optimizer=Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
# 6. TRAINING
print("Starting Training on GPU...")
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=test_generator
)
print("completed")

Starting Training on GPU...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 689s 700ms/step - accuracy: 0.3827 - loss: 1.6153 - val_accuracy: 0.3514 - val_loss: 2.3840
Epoch 2/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 380s 423ms/step - accuracy: 0.5320 - loss: 1.2760 - val_accuracy: 0.3225 - val_loss: 3.0991
Epoch 3/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 376s 419ms/step - accuracy: 0.5636 - loss: 1.1978 - val_accuracy: 0.2707 - val_loss: 2.2297
Epoch 4/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 377s 420ms/step - accuracy: 0.5743 - loss: 1.1609 - val_accuracy: 0.4147 - val_loss: 1.9579
Epoch 5/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 382s 425ms/step - accuracy: 0.5928 - loss: 1.1134 - val_accuracy: 0.5325 - val_loss: 1.2978
Epoch 6/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 375s 418ms/step - accuracy: 0.6019 - loss: 1.0855 - val_accuracy: 0.4702 - val_loss: 1.8030
Epoch 7/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 377s 420ms/step - accuracy: 0.6077 - loss: 1.0748 - val_accuracy: 0.4252 - val_loss: 1.8412
Epoch 8/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 375s 418ms/step - accuracy: 0.6259 -

In [ ]:
# 8. SAVE THE FINAL MODEL
model.save('video_mobilenetv2.h5')
print("\nModel saved successfully")


Model saved successfully
